# 05 — PatchCore Inference: Anomaly Scoring and Maps

**Author:** Sherif Abd El-Rady

**Objective:** Score test images by computing the k-NN distance from each patch to the memory bank. Upsample per-patch scores to full image resolution to produce localisation maps.

## Inference pipeline

```
Test image (224×224)
       │
       ▼
WideResNet-50-2  →  patch embeddings  (28×28 × 1536-D)
       │
       ▼
Project to 128-D (SparseRandomProjection)  →  k-NN distance to memory bank
       │
       ▼
Score map (28×28)  →  bilinear upsample  →  (224×224)
       │
       ├── Image score: max(score_map)
       └── Anomaly map: Gaussian-smoothed score_map
```


In [ ]:
import sys, time
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from defectdetect import FeatureExtractor, MemoryBank
from defectdetect.scorer import AnomalyScorer

ROOT     = Path('..').resolve()
MVTEC    = ROOT / 'data' / 'mvtec'
CATEGORY = 'bottle'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

if not (MVTEC / CATEGORY / 'train' / 'good').exists():
    MVTEC = ROOT / 'data' / 'synthetic'; CATEGORY = ''
    print('Using synthetic dataset')

base     = MVTEC / CATEGORY if CATEGORY else MVTEC
test_dir = base / 'test'
BANK_DIR = ROOT / 'reports' / 'memory_banks'
print('Ready')

In [ ]:
# ── 5.1  Load memory bank and initialise extractor ────────────────────────────
bank_paths = sorted(BANK_DIR.glob('*.pkl'))
if not bank_paths:
    raise FileNotFoundError('No memory bank found. Run Notebook 04 first.')

bank      = MemoryBank.load(bank_paths[-1])
extractor = FeatureExtractor(layers=['layer2','layer3'], input_size=224, device=DEVICE)
scorer    = AnomalyScorer(image_size=(224, 224), gaussian_sigma=4.0)

print(f'Memory bank: {bank.size} entries')

In [ ]:
# ── 5.2  Score all test images ────────────────────────────────────────────────
all_scores, all_labels, all_amaps, all_paths = [], [], [], []
spatial_shape = None

for cls_dir in tqdm(sorted(test_dir.iterdir()), desc='Test classes'):
    if not cls_dir.is_dir(): continue
    label = 0 if cls_dir.name == 'good' else 1
    paths = sorted(cls_dir.glob('*.png')) + sorted(cls_dir.glob('*.jpg'))

    for p in paths:
        img = cv2.imread(str(p))
        if img is None: continue

        batch = extractor.preprocess([img])
        patches, spatial_shape = extractor.extract_patches(batch)
        patch_scores = bank.score(patches, k=1)

        img_score, amap = scorer.compute(patch_scores, spatial_shape, n_images=1)
        all_scores.append(float(img_score[0]))
        all_labels.append(label)
        all_amaps.append(amap[0])
        all_paths.append(p)

scores = np.array(all_scores)
labels = np.array(all_labels)
amaps  = scorer.normalise(np.stack(all_amaps))

print(f'Scored {len(scores)} images  ({(labels==1).sum()} defect / {(labels==0).sum()} normal)')

In [ ]:
# ── 5.3  Score distribution — normal vs defect ────────────────────────────────
normal_scores = scores[labels == 0]
defect_scores = scores[labels == 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(normal_scores, bins=20, color='#2ecc71', alpha=0.8, label=f'Normal (n={len(normal_scores)})')
axes[0].hist(defect_scores, bins=20, color='#e74c3c', alpha=0.8, label=f'Defect (n={len(defect_scores)})')
axes[0].set_xlabel('Anomaly score (max k-NN distance)')
axes[0].set_ylabel('Count')
axes[0].set_title('PatchCore Score Distribution')
axes[0].legend()

# Box plot
axes[1].boxplot([normal_scores, defect_scores], labels=['Normal', 'Defect'],
                patch_artist=True,
                boxprops=dict(facecolor='#ecf0f1'),
                medianprops=dict(color='#e74c3c', linewidth=2))
axes[1].set_ylabel('Anomaly score')
axes[1].set_title('Score Spread — Normal vs Defect')

plt.suptitle('PatchCore anomaly score distributions', fontsize=11)
plt.tight_layout()
plt.show()

overlap = (normal_scores.max() > defect_scores.min())
print(f'Score range — normal: [{normal_scores.min():.3f}, {normal_scores.max():.3f}]')
print(f'Score range — defect: [{defect_scores.min():.3f}, {defect_scores.max():.3f}]')
print(f'Score overlap: {"Yes — threshold selection required" if overlap else "No — perfect separation"}')

In [ ]:
# ── 5.4  Anomaly map visualisation ────────────────────────────────────────────
defect_idxs = np.where(labels == 1)[0][:3]
n_show = len(defect_idxs)

fig, axes = plt.subplots(n_show, 3, figsize=(14, n_show * 3.5))
if n_show == 1: axes = [axes]

for row, idx in enumerate(defect_idxs):
    img_bgr = cv2.imread(str(all_paths[idx]))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    amap    = amaps[idx]  # already normalised
    overlay = scorer.overlay(img_bgr, amap, alpha=0.55)

    axes[row][0].imshow(img_rgb)
    axes[row][0].set_title(f'{all_paths[idx].parent.name}/{all_paths[idx].name}\nscore={scores[idx]:.3f}', fontsize=7)
    axes[row][0].axis('off')

    im = axes[row][1].imshow(amap, cmap='hot', vmin=0, vmax=1)
    axes[row][1].set_title('Anomaly map (normalised)', fontsize=8)
    axes[row][1].axis('off')
    plt.colorbar(im, ax=axes[row][1], fraction=0.046)

    axes[row][2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[row][2].set_title('Heatmap overlay', fontsize=8)
    axes[row][2].axis('off')

plt.suptitle('PatchCore — Anomaly Localisation Maps on Defect Images', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.5  Single-image inference timing ────────────────────────────────────────
test_img = cv2.imread(str(all_paths[defect_idxs[0]]))
N_REPS   = 20

# Warm-up
batch = extractor.preprocess([test_img])
_, _  = extractor.extract_patches(batch)

times = []
for _ in range(N_REPS):
    t0 = time.perf_counter()
    batch = extractor.preprocess([test_img])
    patches, sp = extractor.extract_patches(batch)
    ps   = bank.score(patches, k=1)
    s, m = scorer.compute(ps, sp, n_images=1)
    times.append(time.perf_counter() - t0)

times = np.array(times) * 1000  # ms
print(f'=== Single-Image Inference Timing ({DEVICE}) ===')
print(f'Mean  : {times.mean():.1f} ms')
print(f'Median: {np.median(times):.1f} ms')
print(f'Std   : {times.std():.1f} ms')
print(f'FPS   : {1000/times.mean():.1f}')

## Summary

| Step | Result |
|---|---|
| Score distribution | Defect scores clearly higher than normal |
| Anomaly maps | High scores localise to defect regions |
| Inference speed | *(see Cell 5.5)* |

## Next step

Open `06_evaluation.ipynb` for ROC-AUC, pixel-level metrics, and PatchCore vs baseline comparison.
